# Sesión 2 — Multimodal RAG: del caso práctico al pipeline completo

**Rodrigo López Vera — Data Science, Revolut Perú**  
Curso GenAI Multimodal · Posgrado Data/IA · Mayo 2026

---

### Lo que vas a construir hoy

Al final de esta sesión vas a poder:
1. Indexar documentos normativos en ChromaDB con metadata
2. Hacer retrieval semántico con filtros por metadata
3. Integrar imágenes en el pipeline RAG vía descripción + embedding
4. Construir un pipeline RAG end-to-end con una función
5. Extender un stub de sistema de compliance para el reto final

**Stack:** `google-genai` · `chromadb` · `numpy` · `Pillow`

---
## 00 · Setup

In [29]:
# ChromaDB en Colab puede tener un conflicto con sqlite3 del sistema.
# Si ves un error de sqlite version, ejecuta esto:
# !pip install pysqlite3-binary --quiet
# import sys; sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

!pip install google-genai chromadb numpy Pillow httpx --quiet


[notice] A new release of pip available: 22.2.2 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [30]:
import os
from google import genai
from google.genai import types
from pydantic import BaseModel
from typing import Optional, List
from PIL import Image, ImageDraw
import chromadb
import numpy as np
import json
import io
import re

def load_api_key() -> str:
    try:
        from dotenv import load_dotenv
        load_dotenv()
        key = os.environ.get("GOOGLE_API_KEY")
        if key:
            print("API key cargada desde .env")
            return key
    except ImportError:
        pass
    try:
        from google.colab import userdata
        key = userdata.get("GOOGLE_API_KEY")
        if key:
            print("API key cargada desde Colab Secrets")
            return key
    except Exception:
        pass
    raise EnvironmentError(
        "No se encontró GOOGLE_API_KEY.\n"
        "Local: archivo .env con GOOGLE_API_KEY=AIza...\n"
        "Colab: panel de Secrets (ícono 🔑) con Notebook access activado."
    )

GOOGLE_API_KEY = load_api_key()
# Si ves 429 RESOURCE_EXHAUSTED: crea una API key en un proyecto nuevo en AI Studio.
MODEL = "gemini-3.1-flash-lite-preview"
EMBED_MODEL = "gemini-embedding-2"

client = genai.Client(api_key=GOOGLE_API_KEY)
print("Cliente Gemini inicializado.")

API key cargada desde .env
Cliente Gemini inicializado.


---
## 01 · Arquitectura RAG — el problema que resuelve

```
SIN RAG (lo que hicimos ayer):

  documentos  ──embed en RAM──►  np.argsort  ──►  prompt  ──►  Gemini  ──►  respuesta
  (10 chunks)     cada query                                                       
  
  Problema: no escala. O(n) en memoria y tiempo por cada query.

CON RAG (pipeline de hoy):

  INDEXACIÓN (una sola vez):
  circulares  ──chunk──►  embed  ──►  ChromaDB  ◄── metadata (fuente, artículo, fecha)
  
  CONSULTA (cada vez que llega una query):
  query  ──embed──►  ChromaDB.query  ──►  top-k chunks  ──►  augmented prompt  ──►  Gemini
                     (HNSW index, O(log n))                    + imagen opcional
```

### Dónde falla naive RAG en producción

1. **Chunking malo:** si el chunk corta un artículo a la mitad, pierdes contexto. Solución: chunk por unidad semántica (artículo, párrafo).
2. **Retrieval sin metadata:** recuperas texto pero no sabes de qué norma viene. Solución: guardar metadata en cada chunk.
3. **Top-k sin reranking:** el 1er resultado no siempre es el más útil. Solución: reranking con cross-encoder. *(Lo mencionamos, hoy no lo construimos.)*
4. **Imágenes sin descripción:** `gemini-embedding-2` no entiende pixels. Solución: Gemini describe → embed la descripción.

---
## 02 · Chunking de documentos SPIJ por artículo

Los MD de SPIJ vienen organizados por artículo. Eso hace el chunking trivial: un artículo = un chunk.

In [31]:
def parse_md_to_articles(md_text: str, source_id: str, date: int) -> list[dict]:
    """
    Divide un documento MD por artículo.
    Retorna lista de dicts con: text, source, article, date (int YYYYMM)
    """
    chunks = []
    article_pattern = re.compile(r'^#{1,3}\s*Art[ií]culo\s+[\w]+', re.MULTILINE | re.IGNORECASE)
    matches = list(article_pattern.finditer(md_text))

    if not matches:
        chunks.append({
            "text": md_text.strip(),
            "source": source_id,
            "article": "completo",
            "date": date
        })
        return chunks

    for i, match in enumerate(matches):
        start = match.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(md_text)
        chunk_text = md_text[start:end].strip()
        article_num = match.group(0).strip().split()[-1]
        chunks.append({
            "text": chunk_text,
            "source": source_id,
            "article": article_num,
            "date": date          # int: e.g. 202403
        })

    return chunks


# Test con la circular B-2244-2024
circular_path = "../data/circulares_sbs/circular_B_2244_2024.md"
import os
if not os.path.exists(circular_path):
    circular_path = "data/circulares_sbs/circular_B_2244_2024.md"

with open(circular_path, "r") as f:
    circular_2244 = f.read()

chunks_2244 = parse_md_to_articles(
    md_text=circular_2244,
    source_id="circular_SBS_B_2244_2024",
    date=202403          # int YYYYMM
)

print(f"Chunks generados: {len(chunks_2244)}")
print()
for c in chunks_2244[:3]:
    print(f"  Artículo {c['article']} | source: {c['source']} | fecha: {c['date']}")
    print(f"  {c['text'][:100]}...")
    print()


Chunks generados: 9

  Artículo 1 | source: circular_SBS_B_2244_2024 | fecha: 202403
  ## Artículo 1 — Objeto y ámbito de aplicación

La presente Circular establece los requisitos mínimos...

  Artículo 2 | source: circular_SBS_B_2244_2024 | fecha: 202403
  ## Artículo 2 — Definiciones

Para efectos de la presente Circular se entiende por:

**Operación inu...

  Artículo 3 | source: circular_SBS_B_2244_2024 | fecha: 202403
  ## Artículo 3 — Umbral de reporte de operaciones en efectivo

Las empresas del sistema financiero es...



In [32]:
# Cargar todas las circulares disponibles
import os

BASE_DATA = None
for candidate in [
    "data/circulares_sbs",          # VSCode / local
    "../data/circulares_sbs",       # notebook en subcarpeta
    "/content/circulares_sbs",      # Colab upload manual
]:
    if os.path.isdir(candidate):
        BASE_DATA = candidate
        break

if BASE_DATA is None:
    print("ERROR: No se encontró la carpeta de circulares. Ajusta el path manualmente.")
else:
    print(f"Directorio de datos: {BASE_DATA}")

# date como int YYYYMM — ChromaDB solo acepta int/float en filtros numéricos
circular_configs = [
    {"file": "circular_B_2244_2024.md", "source_id": "circular_SBS_B_2244_2024", "date": 202403},
    {"file": "circular_B_2251_2024.md", "source_id": "circular_SBS_B_2251_2024", "date": 202406},
    {"file": "circular_B_2198_2022.md", "source_id": "circular_SBS_B_2198_2022", "date": 202201},
]

all_chunks = []
for config in circular_configs:
    filepath = os.path.join(BASE_DATA, config["file"])
    if os.path.exists(filepath):
        with open(filepath, "r") as f:
            text = f.read()
        chunks = parse_md_to_articles(text, config["source_id"], config["date"])
        all_chunks.extend(chunks)
        print(f"  {config['file']}: {len(chunks)} chunks")
    else:
        print(f"  [skip] {config['file']} no encontrado")

print(f"\nTotal chunks para indexar: {len(all_chunks)}")


Directorio de datos: ../data/circulares_sbs
  circular_B_2244_2024.md: 9 chunks
  circular_B_2251_2024.md: 9 chunks
  circular_B_2198_2022.md: 5 chunks

Total chunks para indexar: 23


---
## 03 · Indexación en ChromaDB con metadata

ChromaDB es un vector store local. Persiste los datos en disco para que no tengamos que re-indexar en cada sesión.

In [33]:
# Inicializar ChromaDB con persistencia
chroma_client = chromadb.PersistentClient(path="../chroma_db")

# Crear (o conectar a) la colección de circulares
# Si ya existe y quieres empezar de cero: chroma_client.delete_collection("circulares_sbs")
collection = chroma_client.get_or_create_collection(
    name="circulares_sbs",
    metadata={"description": "Circulares SBS - SPIJ Perú"}
)

print(f"Colección: {collection.name}")
print(f"Documentos actuales: {collection.count()}")

Colección: circulares_sbs
Documentos actuales: 23


In [34]:
def embed_texts(texts: list[str]) -> list[list[float]]:
    """Genera embeddings para una lista de textos usando gemini-embedding-2."""
    return [
        client.models.embed_content(model=EMBED_MODEL, contents=text).embeddings[0].values
        for text in texts
    ]


def index_chunks(chunks: list[dict], collection, batch_size: int = 50):
    """Indexa chunks en ChromaDB en batches para evitar límites de API."""
    total = len(chunks)
    indexed = 0

    for i in range(0, total, batch_size):
        batch = chunks[i:i + batch_size]
        texts = [c["text"] for c in batch]
        embeddings = embed_texts(texts)           # list[list[float]] — lo que ChromaDB espera
        ids = [f"{c['source']}__art_{c['article']}" for c in batch]
        metadatas = [
            {"source": c["source"], "article": c["article"], "date": c["date"]}
            for c in batch
        ]
        collection.upsert(ids=ids, embeddings=embeddings, documents=texts, metadatas=metadatas)
        indexed += len(batch)
        print(f"  Indexados: {indexed}/{total}")


if collection.count() == 0:
    print("Indexando chunks...")
    index_chunks(all_chunks, collection)
    print(f"\nColección lista. Total documentos: {collection.count()}")
else:
    print(f"Colección ya tiene {collection.count()} documentos. Saltando indexación.")
    print("Para re-indexar: chroma_client.delete_collection('circulares_sbs') y volver a correr.")


Colección ya tiene 23 documentos. Saltando indexación.
Para re-indexar: chroma_client.delete_collection('circulares_sbs') y volver a correr.


In [35]:
# Inspeccionar la colección antes de hacer queries
sample = collection.peek(limit=3)

print("=== Muestra de documentos indexados ===")
for i in range(len(sample['ids'])):
    print(f"\nID       : {sample['ids'][i]}")
    print(f"Metadata : {sample['metadatas'][i]}")
    print(f"Texto    : {sample['documents'][i][:120]}...")

=== Muestra de documentos indexados ===

ID       : circular_SBS_B_2244_2024__art_1
Metadata : {'date': '2024-03', 'source': 'circular_SBS_B_2244_2024', 'article': '1'}
Texto    : ## Artículo 1 — Objeto y ámbito de aplicación

La presente Circular establece los requisitos mínimos para la detección, ...

ID       : circular_SBS_B_2244_2024__art_2
Metadata : {'article': '2', 'date': '2024-03', 'source': 'circular_SBS_B_2244_2024'}
Texto    : ## Artículo 2 — Definiciones

Para efectos de la presente Circular se entiende por:

**Operación inusual:** Toda transac...

ID       : circular_SBS_B_2244_2024__art_3
Metadata : {'date': '2024-03', 'source': 'circular_SBS_B_2244_2024', 'article': '3'}
Texto    : ## Artículo 3 — Umbral de reporte de operaciones en efectivo

Las empresas del sistema financiero están obligadas a repo...


---
## 04 · Query y retrieval — inspeccionar ANTES de mandar a Gemini

Regla de oro: **valida el retrieval antes de culpar al LLM**. Si el chunk recuperado es incorrecto, la respuesta va a ser mala aunque Gemini sea perfecto.

In [36]:
def retrieve(query: str, n_results: int = 3, where: dict = None) -> list[dict]:
    """
    Recupera los n_results chunks más relevantes para la query.
    Opcionalmente filtra por metadata con `where`.
    Retorna lista de dicts con: text, metadata, distance
    """
    # Embed la query
    query_emb = embed_texts([query])[0]
    
    # Query a ChromaDB
    kwargs = {
        "query_embeddings": [query_emb],
        "n_results": n_results,
        "include": ["documents", "metadatas", "distances"]
    }
    if where:
        kwargs["where"] = where
    
    results = collection.query(**kwargs)
    
    # Formatear resultados
    retrieved = []
    for doc, meta, dist in zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    ):
        retrieved.append({
            "text": doc,
            "metadata": meta,
            "distance": dist,
            "similarity": 1 - dist  # ChromaDB usa distancia L2 por defecto
        })
    
    return retrieved


def print_retrieved(results: list[dict]):
    """Imprime los resultados de retrieval de forma legible."""
    for i, r in enumerate(results):
        print(f"\n--- Resultado {i+1} ---")
        print(f"Fuente   : {r['metadata']['source']}")
        print(f"Artículo : {r['metadata']['article']}")
        print(f"Fecha    : {r['metadata']['date']}")
        print(f"Distancia: {r['distance']:.4f}")
        print(f"Texto    : {r['text'][:200]}...")


# Test de retrieval
test_query = "¿Cuántas transferencias internacionales generan una alerta automática?"
results = retrieve(test_query, n_results=3)
print(f"Query: {test_query}")
print_retrieved(results)

Query: ¿Cuántas transferencias internacionales generan una alerta automática?

--- Resultado 1 ---
Fuente   : circular_SBS_B_2244_2024
Artículo : 5
Fecha    : 2024-03
Distancia: 0.4093
Texto    : ## Artículo 5 — Monitoreo de transacciones y alertas automáticas

Las empresas del sistema financiero deben implementar sistemas de monitoreo transaccional automatizado capaces de:

a) Detectar patron...

--- Resultado 2 ---
Fuente   : circular_SBS_B_2244_2024
Artículo : 4
Fecha    : 2024-03
Distancia: 0.5160
Texto    : ## Artículo 4 — Umbral de reporte de transferencias internacionales

Las empresas supervisadas deberán reportar a la UIF-Perú toda transferencia internacional, sea de envío o recepción, que supere el ...

--- Resultado 3 ---
Fuente   : circular_SBS_B_2251_2024
Artículo : 5
Fecha    : 2024-06
Distancia: 0.6298
Texto    : ## Artículo 5 — Controles de seguridad obligatorios

Los emisores de dinero electrónico deben implementar los siguientes controles:

a) **Autenticación de dos f

---
## 05 · EJERCICIO 1 — 5 queries y evaluación de retrieval

**Objetivo:** evaluar la calidad del retrieval antes de conectar el LLM.  
**Tiempo:** ~40 minutos

Haz 5 queries distintas. Para cada una, responde:
- ¿El top-1 recuperado es el artículo correcto?
- ¿Qué tan cerca está la distancia?
- ¿Hay casos donde el retrieval falla? ¿Por qué crees que falla?

In [37]:
# Queries para evaluar
eval_queries = [
    "¿Cuál es el umbral en USD para reportar transferencias internacionales?",
    "¿Qué es una operación sospechosa según la SBS?",
    "¿Cuánto tiempo tiene el Oficial de Cumplimiento para reportar a la UIF?",
    "¿Cuáles son los límites de saldo en cuentas básicas de billeteras digitales?",
    "¿Qué categoría crediticia corresponde a un deudor con 45 días de atraso?",
]

# TODO: Itera sobre eval_queries, llama a retrieve(), e imprime los resultados
# Anota en comentarios: ¿el retrieval es correcto para cada query?

for i, q in enumerate(eval_queries):
    print(f"\n{'='*60}")
    print(f"QUERY {i+1}: {q}")
    print('='*60)
    # TODO: llamar a retrieve(q, n_results=2) y print_retrieved()


QUERY 1: ¿Cuál es el umbral en USD para reportar transferencias internacionales?

QUERY 2: ¿Qué es una operación sospechosa según la SBS?

QUERY 3: ¿Cuánto tiempo tiene el Oficial de Cumplimiento para reportar a la UIF?

QUERY 4: ¿Cuáles son los límites de saldo en cuentas básicas de billeteras digitales?

QUERY 5: ¿Qué categoría crediticia corresponde a un deudor con 45 días de atraso?


In [38]:
# Asegurar que la colección tiene datos
if collection.count() == 0:
    print("Indexando datos primero...")
    BASE_DATA = None
    for candidate in ["data/circulares_sbs", "genai-multimodal-course/data/circulares_sbs"]:
        if os.path.isdir(candidate):
            BASE_DATA = candidate
            break
    if BASE_DATA:
        configs = [
            ("circular_B_2244_2024.md", "circular_SBS_B_2244_2024", "2024-03"),
            ("circular_B_2251_2024.md", "circular_SBS_B_2251_2024", "2024-06"),
            ("circular_B_2198_2022.md", "circular_SBS_B_2198_2022", "2022-01"),
        ]
        for fname, sid, date in configs:
            path = os.path.join(BASE_DATA, fname)
            if os.path.exists(path):
                with open(path) as f:
                    text = f.read()
                chunks = parse_md_to_articles(text, sid, date)
                embeddings = embed_texts([c["text"] for c in chunks])
                collection.upsert(
                    ids=[f"{c['source']}__art_{c['article']}" for c in chunks],
                    embeddings=embeddings,
                    documents=[c["text"] for c in chunks],
                    metadatas=[{"source": c["source"], "article": c["article"], "date": c["date"]} for c in chunks]
                )
    print(f"Indexado. Total: {collection.count()} docs")

# Evaluación de queries
eval_queries = [
    ("¿Cuál es el umbral en USD para reportar transferencias internacionales?",
     "Art. 4 — circular_SBS_B_2244_2024"),
    ("¿Qué es una operación sospechosa según la SBS?",
     "Art. 2 — circular_SBS_B_2244_2024"),
    ("¿Cuánto tiempo tiene el Oficial de Cumplimiento para reportar a la UIF?",
     "Art. 7 — circular_SBS_B_2244_2024"),
    ("¿Cuáles son los límites de saldo en cuentas básicas de billeteras digitales?",
     "Art. 3 — circular_SBS_B_2251_2024"),
    ("¿Qué categoría crediticia corresponde a un deudor con 45 días de atraso?",
     "Art. 2 — circular_SBS_B_2198_2022"),
]

print(f"{'Query':<55} {'Top-1 recuperado':<45} {'Esperado':<45} {'OK?'}")
print("-" * 160)

for query, expected in eval_queries:
    results = retrieve(query, n_results=2)
    if results:
        top1 = f"Art. {results[0]['metadata']['article']} — {results[0]['metadata']['source']}"
        ok = "✓" if results[0]['metadata']['article'] in expected else "✗"
    else:
        top1 = "[sin resultados]"
        ok = "✗"
    print(f"{query[:54]:<55} {top1[:44]:<45} {expected[:44]:<45} {ok}")

Query                                                   Top-1 recuperado                              Esperado                                      OK?
----------------------------------------------------------------------------------------------------------------------------------------------------------------
¿Cuál es el umbral en USD para reportar transferencias  Art. 4 — circular_SBS_B_2244_2024             Art. 4 — circular_SBS_B_2244_2024             ✓
¿Qué es una operación sospechosa según la SBS?          Art. 2 — circular_SBS_B_2244_2024             Art. 2 — circular_SBS_B_2244_2024             ✓
¿Cuánto tiempo tiene el Oficial de Cumplimiento para r  Art. 7 — circular_SBS_B_2244_2024             Art. 7 — circular_SBS_B_2244_2024             ✓
¿Cuáles son los límites de saldo en cuentas básicas de  Art. 3 — circular_SBS_B_2251_2024             Art. 3 — circular_SBS_B_2251_2024             ✓
¿Qué categoría crediticia corresponde a un deudor con   Art. 2 — circular_SBS_B_2198_20

---
## 06 · Metadata filtering

ChromaDB permite filtrar por metadata antes de hacer el retrieval.  
Útil para: "solo circulares de 2024", "solo de esta fuente específica", etc.

In [39]:
# Filtro por fecha: solo circulares del 2024 en adelante
# ChromaDB solo acepta int o float en operadores numéricos ($gte, $lte, etc.)
# Usamos formato YYYYMM como entero: 202401, 202403, etc.
query = "¿Cuáles son los límites de transferencia para billeteras digitales?"

# Sin filtro
results_all = retrieve(query, n_results=3)
print("=== Sin filtro ===")
for r in results_all:
    print(f"  {r['metadata']['source']} | art. {r['metadata']['article']} | fecha: {r['metadata']['date']}")

print()

# Con filtro: solo circulares con date >= 202401 (enero 2024)
results_2024 = retrieve(
    query,
    n_results=3,
    where={"date": {"$gte": 202401}}
)
print("=== Solo circulares >= 202401 (enero 2024) ===")
for r in results_2024:
    print(f"  {r['metadata']['source']} | art. {r['metadata']['article']} | fecha: {r['metadata']['date']}")


=== Sin filtro ===
  circular_SBS_B_2251_2024 | art. 3 | fecha: 2024-06
  circular_SBS_B_2251_2024 | art. 4 | fecha: 2024-06
  circular_SBS_B_2251_2024 | art. 2 | fecha: 2024-06

=== Solo circulares >= 202401 (enero 2024) ===


In [40]:
# Filtro por fuente específica
results_2244 = retrieve(
    "¿Cuáles son las sanciones por incumplimiento?",
    n_results=2,
    where={"source": {"$eq": "circular_SBS_B_2244_2024"}}
)
print("=== Solo circular B-2244-2024 ===")
print_retrieved(results_2244)

=== Solo circular B-2244-2024 ===

--- Resultado 1 ---
Fuente   : circular_SBS_B_2244_2024
Artículo : 8
Fecha    : 2024-03
Distancia: 0.5544
Texto    : ## Artículo 8 — Régimen de sanciones

El incumplimiento de las disposiciones de la presente Circular será sancionado conforme al Reglamento de Sanciones de la SBS, con multas que oscilan entre **5 y 2...

--- Resultado 2 ---
Fuente   : circular_SBS_B_2244_2024
Artículo : 7
Fecha    : 2024-03
Distancia: 0.6992
Texto    : ## Artículo 7 — Obligaciones del Oficial de Cumplimiento

El Oficial de Cumplimiento es el responsable directo ante la SBS del cumplimiento de las presentes disposiciones. Sus obligaciones incluyen:

...


---
## 07 · Multimodal RAG — imágenes en el pipeline

El problema: `gemini-embedding-2` solo acepta texto.  
La solución: **Gemini describe la imagen → embed la descripción → guardar metadata con referencia a la imagen**.

No es un workaround: es exactamente lo que se hace en producción cuando no se tiene acceso a multimodal embeddings propietarios.

In [41]:
# Colección separada para imágenes
image_collection = chroma_client.get_or_create_collection(
    name="vouchers_financieros",
    metadata={"description": "Vouchers e imágenes financieras ficticias"}
)
print(f"Colección de imágenes: {image_collection.count()} documentos")

Colección de imágenes: 0 documentos


In [42]:
def describe_image(image: Image.Image, context: str = "") -> str:
    """
    Usa Gemini para generar una descripción textual de una imagen.
    Esta descripción es lo que se va a embedear e indexar.
    """
    prompt = f"""
Describe este documento financiero de forma detallada y estructurada.
Incluye: tipo de documento, entidad emisora, montos, monedas, fechas, 
partes involucradas (origen y destino), concepto/propósito, y estado.
Si hay datos ilegibles, indícalo. Responde solo en texto plano, sin markdown.
{f'Contexto adicional: {context}' if context else ''}
"""
    response = client.models.generate_content(
        model=MODEL,
        contents=[image, prompt],
        config=types.GenerateContentConfig(temperature=0.0)
    )
    return response.text


def index_image(image: Image.Image, image_id: str, metadata: dict, image_collection):
    """
    Describe la imagen con Gemini, embedea la descripción, y la indexa en ChromaDB.
    La metadata incluye la referencia a la imagen original.
    """
    # Paso 1: Gemini describe la imagen
    description = describe_image(image)
    print(f"Descripción generada para {image_id}:")
    print(f"  {description[:150]}...")
    
    # Paso 2: Embed la descripción
    embedding = embed_texts([description])[0]
    
    # Paso 3: Indexar con metadata (incluyendo la descripción para poder mostrarla)
    full_metadata = {**metadata, "description": description[:500]}  # ChromaDB limita metadata
    
    image_collection.upsert(
        ids=[image_id],
        embeddings=[embedding],
        documents=[description],
        metadatas=[full_metadata]
    )
    
    return description

In [44]:
# Definir las imágenes a indexar
# En producción estas vendrían de S3, GCS, etc.
# Aquí las recreamos en memoria como en la sesión 1

def make_voucher_pil(title, lines, color=(0, 102, 204)):
    """Genera un voucher PIL simple."""
    img = Image.new("RGB", (480, 400), color="white")
    draw = ImageDraw.Draw(img)
    draw.rectangle([0, 0, 480, 60], fill=color)
    draw.text((240, 30), title, fill="white", anchor="mm")
    y = 80
    for key, value in lines:
        draw.text((20, y), f"{key}:", fill=(100, 100, 100))
        draw.text((460, y), value, fill=(20, 20, 20), anchor="ra")
        draw.line([20, y + 18, 460, y + 18], fill=(220, 220, 220), width=1)
        y += 32
    return img

images_to_index = [
    {
        "id": "voucher_yape_001",
        "image": make_voucher_pil(
            "YAPE - TRANSFERENCIA",
            [("Fecha", "02/05/2024"), ("Monto", "S/ 350.00"),
             ("De", "Luis Quispe"), ("Para", "Ana Torres"),
             ("Concepto", "Pago cuota depa"), ("Estado", "EXITOSO")],
            color=(102, 45, 145)
        ),
        "metadata": {"type": "yape_p2p", "date": 20240502, "amount_pen": "350", "currency": "PEN"}
    },
    {
        "id": "voucher_bbva_swift_003",
        "image": make_voucher_pil(
            "BBVA - SWIFT INTERNACIONAL",
            [("Fecha", "28/04/2024"), ("Monto", "USD 15,000.00"),
             ("Banco destino", "Santander Uruguay"), ("Beneficiario", "C. Mendoza EIRL"),
             ("Proposito", "Pago proveedor"), ("Estado", "EN PROCESO")],
            color=(0, 40, 130)
        ),
        "metadata": {"type": "swift_internacional", "date": 20240428, "amount_usd": "15000", "currency": "USD"}
    },
    {
        "id": "voucher_interbank_deposito_004",
        "image": make_voucher_pil(
            "INTERBANK - DEPOSITO EFECTIVO",
            [("Fecha", "03/05/2024"), ("Monto", "S/ 12,800.00"),
             ("Titular", "Rodriguez V. Carmen"), ("Agencia", "La Victoria"),
             ("Concepto", "Venta inmueble"), ("Estado", "COMPLETADO")],
            color=(0, 156, 68)
        ),
        "metadata": {"type": "deposito_efectivo", "date": 20240503, "amount_pen": "12800", "currency": "PEN"}
    },
]

# Indexar si la colección está vacía
if image_collection.count() == 0:
    print("Indexando imágenes...\n")
    for img_data in images_to_index:
        index_image(
            image=img_data["image"],
            image_id=img_data["id"],
            metadata=img_data["metadata"],
            image_collection=image_collection
        )
        print()
    print(f"Total imágenes indexadas: {image_collection.count()}")
else:
    print(f"Colección de imágenes ya tiene {image_collection.count()} documentos.")

Colección de imágenes ya tiene 2 documentos.


---
## 08 · Pipeline end-to-end — función `rag_query()`

Aquí conectamos todo: retrieval de texto + retrieval de imágenes + generación con Gemini.

In [45]:
class RAGResponse(BaseModel):
    answer: str
    sources: list[str]
    confidence_note: str


def rag_query(
    question: str,
    image: Optional[Image.Image] = None,
    n_text_chunks: int = 3,
    n_image_results: int = 2,
    date_filter: Optional[int] = None,   # int YYYYMM, ej: 202401
    verbose: bool = False
) -> RAGResponse:
    """
    Pipeline RAG completo.

    Parámetros:
        question       : pregunta del analista
        image          : voucher/imagen opcional para incluir en el contexto
        n_text_chunks  : cuántos fragmentos de circulares recuperar
        n_image_results: cuántas imágenes similares recuperar del vector store
        date_filter    : filtro de fecha mínima como int YYYYMM (ej: 202401)
        verbose        : imprimir chunks recuperados
    """

    # --- PASO 1: Retrieval de texto normativo ---
    where = {"date": {"$gte": date_filter}} if date_filter else None
    text_results = retrieve(question, n_results=n_text_chunks, where=where)

    if verbose:
        print("=== Fragmentos normativos recuperados ===")
        for r in text_results:
            print(f"  [{r['metadata']['source']} | Art. {r['metadata']['article']}]")
            print(f"  {r['text'][:100]}...")
        print()

    # --- PASO 2: Retrieval de imágenes similares (si hay colección) ---
    image_context = ""
    if image_collection.count() > 0:
        q_emb = embed_texts([question])[0]
        img_raw = image_collection.query(
            query_embeddings=[q_emb],
            n_results=min(n_image_results, image_collection.count()),
            include=["documents", "metadatas", "distances"]
        )
        if img_raw['documents'][0]:
            image_context = "\n\nCONTEXTO DE IMÁGENES SIMILARES INDEXADAS:\n"
            for doc, meta in zip(img_raw['documents'][0], img_raw['metadatas'][0]):
                image_context += f"- [{meta.get('type', 'imagen')} | {meta.get('date', '')}]: {doc[:200]}\n"

    # --- PASO 3: Construir el prompt de augmentation ---
    normativa_context = "\n\n---\n\n".join([
        f"[{r['metadata']['source']} | Artículo {r['metadata']['article']}]\n{r['text']}"
        for r in text_results
    ])
    sources = [
        f"{r['metadata']['source']} · Art. {r['metadata']['article']}"
        for r in text_results
    ]
    augmented_prompt = f"""
Eres un analista de compliance del sistema financiero peruano.
Responde la pregunta ÚNICAMENTE basándote en la normativa SBS proporcionada.
Cita el artículo y la circular exactos. Si la normativa no cubre la pregunta, dilo.

=== NORMATIVA SBS RECUPERADA ===
{normativa_context}
{image_context}

=== PREGUNTA DEL ANALISTA ===
{question}
"""

    # --- PASO 4: Llamar a Gemini con la imagen si se proporcionó ---
    contents = [augmented_prompt] if image is None else [image, augmented_prompt]
    response = client.models.generate_content(
        model=MODEL,
        contents=contents,
        config=types.GenerateContentConfig(
            temperature=0.0,
            max_output_tokens=600,
            response_mime_type="application/json",
            response_schema=RAGResponse
        )
    )
    parsed = json.loads(response.text)
    if not parsed.get("sources"):
        parsed["sources"] = sources
    return RAGResponse(**parsed)


print("Pipeline RAG definido.")


Pipeline RAG definido.


In [46]:
# Test del pipeline completo
question = "Un cliente realizó 3 transferencias internacionales en 48 horas por USD 45,000 hacia Panamá e Islas Caimán. ¿Requiere reporte a la UIF?"

result = rag_query(question, verbose=True)

print("=" * 60)
print("RESPUESTA:")
print(result.answer)
print()
print("FUENTES:")
for s in result.sources:
    print(f"  - {s}")
print()
print(f"NOTA DE CONFIANZA: {result.confidence_note}")

=== Fragmentos normativos recuperados ===
  [circular_SBS_B_2244_2024 | Art. 4]
  ## Artículo 4 — Umbral de reporte de transferencias internacionales

Las empresas supervisadas deber...
  [circular_SBS_B_2244_2024 | Art. 5]
  ## Artículo 5 — Monitoreo de transacciones y alertas automáticas

Las empresas del sistema financier...
  [circular_SBS_B_2244_2024 | Art. 3]
  ## Artículo 3 — Umbral de reporte de operaciones en efectivo

Las empresas del sistema financiero es...



ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [47]:
# Pipeline con imagen: el voucher del BBVA + pregunta de compliance
voucher_bbva = make_voucher_pil(
    "BBVA - SWIFT INTERNACIONAL",
    [("Fecha", "28/04/2024"), ("Monto", "USD 15,000.00"),
     ("Banco destino", "Santander Uruguay"), ("Beneficiario", "C. Mendoza EIRL"),
     ("Proposito", "Pago proveedor"), ("Estado", "EN PROCESO")],
    color=(0, 40, 130)
)

question_img = "Analiza este voucher. ¿La operación que muestra requiere reporte a la UIF-Perú? ¿Bajo qué artículo?"

result_img = rag_query(question_img, image=voucher_bbva, verbose=False)

print("=== RAG + IMAGEN ===")
print(f"Respuesta: {result_img.answer}")
print(f"Fuentes  : {result_img.sources}")
print(f"Nota     : {result_img.confidence_note}")

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

---
## 09 · EJERCICIO 2 — Retrieval mixto texto + imagen

**Objetivo:** indexar 2 imágenes adicionales y hacer una query que las recupere junto con texto normativo.

1. Crea un voucher PIL del depósito en efectivo de S/ 12,800 en Interbank (los datos están en `images_to_index`)
2. Indexa esa imagen con `index_image()`
3. Haz la query: `"¿Un depósito en efectivo de S/ 12,800 requiere reporte? ¿Cuál es el umbral?"`
4. Usa `rag_query()` con `verbose=True` y analiza qué fragmentos recupera

In [48]:
# TODO: Ejercicio 2
# 1. Crea el voucher de depósito en efectivo Interbank
# 2. Indexa con index_image()
# 3. Query mixta con rag_query(verbose=True)
# 4. Discute: ¿recuperó el artículo correcto? ¿Por qué sí o no?

# TODO: implementar

---
## 10 · RETO — Financial Profile Interrogator

**Tiempo:** 20 min construcción + 20 min presentaciones  
**Grupos:** 3-4 personas

### Escenario

> Construir un sistema que, dado un perfil de cliente (imagen o texto) y una pregunta de compliance, responda citando el artículo exacto de la normativa SBS que aplica.

### Consigna

El stub de abajo ya tiene el pipeline funcionando. Tu trabajo es **extenderlo** de una de estas formas (elige una):

**Opción A — Logging de queries:**  
Agrega un log de cada query realizada: timestamp, question, top-1 source, respuesta. Guárdalo como CSV.

**Opción B — Filtro por tipo de operación:**  
Agrega un parámetro `operation_type` que filtre los chunks recuperados según el tipo de circular (LA/FT, dinero electrónico, crédito).

**Opción C — Reranking simple:**  
Después del retrieval, reordena los chunks por longitud (chunks más largos = más contexto). ¿Mejora la respuesta?

**Opción D — Respuesta comparativa:**  
Para la misma query, ejecuta el pipeline con y sin imagen. Compara las respuestas. ¿La imagen agrega información?

### Query de ejemplo que el sistema debe poder responder:

> "Este cliente tiene 3 transferencias internacionales en 48 horas por un total de S/. 45,000. ¿Requiere reporte a la UIF según la normativa vigente?"

In [ ]:
# STUB — Financial Profile Interrogator
# Este código ya corre. Tu tarea: extenderlo según la opción elegida.

import csv
from datetime import datetime


class FinancialProfileInterrogator:
    """
    Sistema de compliance para interrogar perfiles financieros
    contra la normativa SBS.
    """

    def __init__(self, verbose: bool = False):
        self.verbose = verbose
        self.query_log = []  # Opción A: log de queries

    def analyze(
        self,
        client_profile: str,
        question: str,
        voucher_image: Optional[Image.Image] = None,
        date_filter: Optional[int] = None   # int YYYYMM, ej: 202401
    ) -> dict:
        """
        Analiza un perfil de cliente contra la normativa.

        Parámetros:
            client_profile : descripción textual del perfil del cliente
            question       : pregunta de compliance
            voucher_image  : imagen de voucher (opcional)
            date_filter    : int YYYYMM para filtrar solo normas a partir de esa fecha
        """
        full_question = f"""
PERFIL DEL CLIENTE:
{client_profile}

PREGUNTA DE COMPLIANCE:
{question}
"""
        result = rag_query(
            question=full_question,
            image=voucher_image,
            n_text_chunks=4,
            date_filter=date_filter,
            verbose=self.verbose
        )
        # TODO (Opción A): agregar al log
        # self.query_log.append({...})
        return {
            "answer": result.answer,
            "sources": result.sources,
            "confidence_note": result.confidence_note
        }

    # TODO (Opción A): implementar save_log(filename)
    # def save_log(self, filename: str = "query_log.csv"):
    #     pass


interrogator = FinancialProfileInterrogator(verbose=True)

client_profile = """
Tipo de cliente: persona natural
Actividad declarada: exportación de textiles
Antigüedad en el banco: 8 meses
Historial: sin alertas previas
Movimiento reciente: 3 transferencias internacionales en 48 horas
  - USD 15,000 → Banco en Panamá
  - USD 12,000 → Banco en Islas Caimán
  - USD 18,000 → Banco en Panamá
  - Total: USD 45,000
"""

question = "¿Requiere reporte a la UIF según la normativa vigente? ¿Bajo qué artículo?"

result = interrogator.analyze(
    client_profile=client_profile,
    question=question,
    date_filter=202401        # solo normas desde enero 2024
)

print("\n" + "=" * 60)
print("RESULTADO DEL ANÁLISIS")
print("=" * 60)
print(f"\nRespuesta:\n{result['answer']}")
print(f"\nFuentes: {result['sources']}")
print(f"\nNota: {result['confidence_note']}")


In [ ]:
# ESPACIO DE TRABAJO DEL GRUPO
# Implementar aquí la extensión elegida (Opción A, B, C o D)

# TODO: implementar

---
## Debrief final — ¿Qué haría diferente en producción?

**Lo que construimos hoy vs producción real:**

| Hoy | Producción |
|-----|------------|
| ChromaDB local | Vertex AI Vector Search o Pinecone |
| Chunking por artículo | Chunking + reranking con cross-encoder |
| Top-k fijo | Hybrid search (dense + BM25 sparse) |
| Imagen → descripción → embed | Multimodal embeddings con Vertex AI |
| Sin observabilidad | LangSmith / Vertex AI Eval para monitoreo |

**La arquitectura que usamos en BCP:**
- Databricks para la ingesta y preprocessing
- Unity Catalog para linaje de datos y permisos
- Context offloading para conversaciones largas con muchos documentos
- Evaluación mensual de respuestas con panel de revisores humanos

**Lo más importante que aprendí construyendo esto en producción:**  
El 80% del trabajo no es el LLM. Es el chunking, la metadata, la arquitectura, la evaluación del retrieval, y el feedback loop con usuarios reales.